### Import and Loading

In [9]:
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
df = pd.read_csv("../data/cleaned_timeseries.csv")

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

train_df = df.loc["2013-01-02":"2013-12-31"].copy()
test_df  = df.loc["2014-01-01":"2014-03-31"].copy()

print("Train:", train_df.index.min(), "to", train_df.index.max(), "| rows:", len(train_df))
print("Test :", test_df.index.min(), "to", test_df.index.max(), "| rows:", len(test_df))
print("Target missing (train/test):", train_df["unit_sales"].isna().sum(), test_df["unit_sales"].isna().sum())

Train: 2013-01-02 00:00:00 to 2013-12-31 00:00:00 | rows: 364
Test : 2014-01-01 00:00:00 to 2014-03-31 00:00:00 | rows: 90
Target missing (train/test): 0 0


In [3]:
## Create lag features (on full df first)
# Lag features
df["lag_1"] = df["unit_sales"].shift(1)
df["lag_7"] = df["unit_sales"].shift(7)
df["lag_14"] = df["unit_sales"].shift(14)

# Rolling features (based only on past values)
df["rolling_mean_7"] = df["unit_sales"].shift(1).rolling(7).mean()
df["rolling_std_7"]  = df["unit_sales"].shift(1).rolling(7).std()

# Calendar features (numeric)
df["day_of_week"] = df.index.dayofweek
df["month"] = df.index.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)


In [4]:
## Drop initial NaNs created by lagging
feature_cols = [
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_std_7"
]

df_ml = df.dropna(subset=feature_cols).copy()

print("Rows after ML feature cleaning:", len(df_ml))

Rows after ML feature cleaning: 440


In [5]:
train_ml = df_ml.loc["2013-01-02":"2013-12-31"].copy()
test_ml  = df_ml.loc["2014-01-01":"2014-03-31"].copy()

print("Train rows:", len(train_ml))
print("Test rows :", len(test_ml))
print("Train start:", train_ml.index.min(), "end:", train_ml.index.max())
print("Test start :", test_ml.index.min(), "end:", test_ml.index.max())

Train rows: 350
Test rows : 90
Train start: 2013-01-16 00:00:00 end: 2013-12-31 00:00:00
Test start : 2014-01-01 00:00:00 end: 2014-03-31 00:00:00


#### Train starts on 16.01. because lag_14 needs 14 previous days + rolling features need history, so early January gets dropped.

### Linear Regression baseline

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

features = [
    "lag_1", "lag_7", "lag_14",
    "rolling_mean_7", "rolling_std_7",
    "day_of_week", "month", "is_weekend",
    "is_holiday", "is_sunny", "dcoilwtico"
]

X_train = train_ml[features]
y_train = train_ml["unit_sales"]

X_test = test_ml[features]
y_test = test_ml["unit_sales"]

lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))

# sMAPE (safe with zeros)
smape_lr = 100 * np.mean(2 * np.abs(pred_lr - y_test) / (np.abs(y_test) + np.abs(pred_lr) + 1e-8))

print("LinearRegression MAE :", mae_lr)
print("LinearRegression RMSE:", rmse_lr)
print("LinearRegression sMAPE:", smape_lr)

LinearRegression MAE : 101.11775622556578
LinearRegression RMSE: 147.61966396544955
LinearRegression sMAPE: 22.79886316661221


* Linear Regression with engineered features is very competitive with Prophet.
* It even achieves the lowest RMSE so far.
* This shows the lag + rolling + exogenous features are strong.

### Random Forest (non-linear model)

In [7]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
smape_rf = 100 * np.mean(2 * np.abs(pred_rf - y_test) / (np.abs(y_test) + np.abs(pred_rf) + 1e-8))

print("RandomForest MAE :", mae_rf)
print("RandomForest RMSE:", rmse_rf)
print("RandomForest sMAPE:", smape_rf)

RandomForest MAE : 104.36322222222225
RandomForest RMSE: 152.0304139207467
RandomForest sMAPE: 23.220902658415017


#### Random Forest is worse than Linear Regression because:
* The time series is mostly linear and autoregressive
* Lag structure dominates
* Complex non-linear trees do not add much value

### XGBoost

In [8]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
smape_xgb = 100 * np.mean(2 * np.abs(pred_xgb - y_test) / (np.abs(y_test) + np.abs(pred_xgb) + 1e-8))

print("XGBoost MAE :", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)
print("XGBoost sMAPE:", smape_xgb)

XGBoost MAE : 110.98184204101562
XGBoost RMSE: 162.0784663215827
XGBoost sMAPE: 24.265748603114893


#### Feature-engineered machine learning models achieve competitive performance compared to classical statistical models. Linear Regression with lag and rolling features performs best among ML approaches. However, seasonal statistical models such as Prophet and ETS still slightly outperform ML models, indicating that the time series structure is primarily linear with strong seasonal patterns rather than complex nonlinear relationships.